# Séance 10 — Déploiement : une API de scoring en temps réel
*Session 10 — Deployment: a real-time scoring API*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/10_deploiement_api.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ○ démonstration |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ● séance complète |

**À la fin de cette séance, vous saurez :**

1. décrire l'architecture d'un service de scoring (modèle + règles + seuils) ;
2. appeler une API FastAPI et interpréter sa réponse ;
3. reproduire la logique de décision BLOQUER / REVOIR / APPROUVER ;
4. mesurer la latence d'un service par rapport à un objectif (100 ms).

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("entrainer")
from fastapi.testclient import TestClient
from fraude360.serving import api
client = TestClient(api.app)
client.__enter__()                       # démarre le service (charge le champion et les règles)
print(client.get("/health").json())
print(client.get("/modele").json())

In [ ]:
regles = client.get("/regles").json()
pd.DataFrame(regles if isinstance(regles, list) else regles.get("regles", regles)).head(12)

In [ ]:
gold = table("gold", "caracteristiques")
etat = api.ETAT
variables = etat["variables"]

def demande(ligne):
    import json
    return {"transaction_id": str(ligne["transaction_id"]), "variables": json.loads(ligne[variables].to_json())}

ex = gold.sample(1, random_state=3).iloc[0]
reponse = client.post("/score", json=demande(ex)).json()
reponse

**En production**, le même service tourne dans un conteneur (`Dockerfile`, `uvicorn fraude360.serving.api:app`) ou sur
Databricks Model Serving. Les variables sont fournies par la couche Gold / un *feature store* en ligne.

### Exercice 10.1 — 🟢 Tous les parcours

Écrivez `decider(score_modele, score_regles, regles, etat)` :
1. `BLOQUER` si `score_modele >= etat["seuil_blocage"]`;
2. sinon `BLOQUER` si une règle déclenchée est de gravité « elevee » **et** `score_regles >= 0.5`;
3. sinon `REVOIR` si `score_modele >= etat["seuil_revue"]` ou `score_regles >= 0.35`;
4. sinon `APPROUVER`.

Les règles graves : `{r.identifiant for r in etat["moteur"].regles if r.gravite == "elevee"}`. Renvoyez seulement la décision.

In [ ]:
# VOTRE CODE ICI
def decider(score_modele, score_regles, regles, etat):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_rng = np.random.default_rng(1)
_ids = [r.identifiant for r in etat["moteur"].regles]
_ok = True
for _ in range(300):
    s, r = float(_rng.random() ** 3), float(_rng.random())
    rg = list(_rng.choice(_ids, size=_rng.integers(0, 3), replace=False))
    _ok &= decider(s, r, rg, etat) == api.decider(s, r, rg, etat)[0]
verifier(_ok, "Même décision que le service sur 300 cas aléatoires")

### Exercice 10.2 — 🟣 Gestion + Informatique

Envoyez 200 transactions au service (une requête chacune) et calculez `latences` (en millisecondes, mesurées
avec `time.perf_counter()` côté client). Affichez les percentiles 50 et 95. L'objectif est `cfg["service"]["latence_cible_ms"]`.

In [ ]:
# VOTRE CODE ICI
import time
latences = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(len(latences) == 200 and np.percentile(latences, 95) < 2000)

### Exercice 10.3 — 🟢 Tous les parcours

Trouvez dans `gold` une **fraude** avec un montant supérieur à 100 $, envoyez-la au service et stockez la
réponse dans `rep_fraude`. Quelle est la décision, et quel motif le service donne-t-il ?

In [ ]:
# VOTRE CODE ICI
rep_fraude = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(rep_fraude["decision"] in {"BLOQUER", "REVOIR", "APPROUVER"} and "motif" in rep_fraude)

In [ ]:
client.__exit__(None, None, None)    # arrête le service

**❓ Question (🔶 Gestion — décision d'affaires) —** Le service répond en 30 ms mais bloque 2 % des transactions d'un commerçant important. Quelles informations fournissez-vous au service à la clientèle pour traiter les plaintes ?

*Votre réponse ici.*

## Bilan

- Un service de scoring combine modèle, règles expertes et seuils, et explique chaque décision.
- Le modèle est chargé par alias depuis le registre : on promeut sans redéployer.
- Latence, journalisation et rechargement à chaud font partie de la définition de « terminé ».

**Pour aller plus loin :**

- `scripts/demo_api.py` : démarre le service avec uvicorn et envoie des requêtes HTTP réelles.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*